# Assignment 4- Christopher Felesena and Ryan Brock (Group19)

## Problem 1: Inventory Planning with Monte Carlo Simulation (40 Points)

A campus bookstore must decide how many university-branded hoodies to stock for an upcoming sales week. We simulate uncertain customer demand, evaluate the financial outcome of each inventory level, and recommend the best stocking strategy.

| Parameter | Value |
|---|---|
| Average weekly demand | 200 hoodies |
| Standard deviation of demand | 40 hoodies |
| Selling price per hoodie | \$45 |
| Purchase cost per hoodie | \$25 |
| Salvage value of unsold hoodies | \$0 |

**Inventory levels considered:** 180, 220, and 260 hoodies.

In [1]:
import numpy as np
import pandas as pd

# Initialize bookstore demand expectations and test stocking levels
bookstore_information = {"Average weekly demand": 200,
                         "Standard Deviation": 40,
                         "Sell Price": 45,
                         "Purchase Price": 25,
                         "Salvage Price": 0}

stock_levels = np.array([180,
                         220,
                         260])

sample_size = 1000

## Part A: Simulating Customer Demand (10 Points)

### A.1
Create a NumPy random number generator using a seed of 42.

In [2]:
# A.1
rng = np.random.default_rng(42)

### A.2
Generate **1,000 simulated weekly demand values** using a normal distribution with the mean and standard deviation provided above.

In [3]:
# A.2
simulated_demand = rng.normal(bookstore_information["Average weekly demand"],
                              bookstore_information["Standard Deviation"],
                              size=sample_size)
print("Number of simulated weeks:", simulated_demand.size)

Number of simulated weeks: 1000


### A.3
Since customer demand cannot be negative, replace negative simulated values with zero.

In [4]:
# A.3
print("Number of negative demand values replaced:", np.sum(simulated_demand < 0))
simulated_demand = np.maximum(np.rint(simulated_demand), 0).astype(int)

Number of negative demand values replaced: 0


### A.4
Calculate and display the following statistics for simulated demand:
- Mean demand
- Minimum and maximum demand
- Standard deviation of demand
- Percentage of simulated weeks in which demand exceeds 220 hoodies

In [5]:
# A.4
print("For sample size: " + str(sample_size))
print("Mean: " + str(round(np.mean(simulated_demand), 2)))
print("Minimum demand: " + str(np.min(simulated_demand)))
print("Maximum demand: " + str(np.max(simulated_demand)))
print("Standard deviation: " + str(round(np.std(simulated_demand), 2)))
print("Percent of weeks over 220 hoodies demanded: " + str(round(np.mean(simulated_demand > 220)*100, 2)) + "%")

For sample size: 1000
Mean: 198.86
Minimum demand: 54
Maximum demand: 327
Standard deviation: 39.55
Percent of weeks over 220 hoodies demanded: 27.4%


## Part B: Evaluating Inventory Decisions (15 Points)

Evaluate each of the three inventory levels (180, 220, and 260 hoodies) using the **same 1,000 simulated demand values** so the comparison is consistent.

### B.1
Calculate the number of hoodies sold in each simulated week. Sales cannot exceed inventory or customer demand.

In [6]:
# B.1
inventory_sold = {}
for stock in stock_levels:
    inventory_sold[stock] = np.minimum(simulated_demand, stock)

### B.2
Calculate the number of unsold hoodies and the number of customers whose demand could not be fulfilled.

In [7]:
# B.2
unsold_inventory = {}
unfilled_sales = {}
for stock in stock_levels:
    unsold_inventory[stock] = np.maximum(stock - simulated_demand, 0)
    unfilled_sales[stock] = -1*np.minimum(stock - simulated_demand, 0)

### B.3
Calculate weekly profit using:

**Profit = (Units Sold × Selling Price) − (Units Stocked × Purchase Cost)**

In [8]:
# B.3
profit = {}
for stock in stock_levels:
    profit[stock] = (inventory_sold[stock]*bookstore_information["Sell Price"]
                     - stock*bookstore_information["Purchase Price"]
                     + unsold_inventory[stock]*bookstore_information["Salvage Price"])

### B.4
Calculate the following:
- Average weekly profit
- Average number of unsold hoodies
- Average number of unfulfilled hoodie sales
- Percentage of weeks with a stockout (demand exceeds inventory)

In [9]:
# B.4
results = []

print("Testing stock levels")

for stock in stock_levels:
    results.append({"Stock Level": stock,
                    "Profit": np.mean(profit[stock]),
                    "Unsold Inventory": np.mean(unsold_inventory[stock]),
                    "Unfulfilled Sales": np.mean(unfilled_sales[stock]),
                    "Stockout Percent": np.mean(simulated_demand > stock)*100})

    print("For stock level: " + str(stock))
    print("Average weekly profit: " + str(np.mean(profit[stock])))
    print("Average unsold hoodies: " + str(np.mean(unsold_inventory[stock])))
    print("Average unfulfilled demand: " + str(np.mean(unfilled_sales[stock])))
    print("Percent of weeks with a stockout: " + str(round(np.mean(simulated_demand > stock)*100, 2)) + "%.")
    print("")

Testing stock levels
For stock level: 180
Average weekly profit: 3229.92
Average unsold hoodies: 8.224
Average unfulfilled demand: 27.079
Percent of weeks with a stockout: 69.9%.

For stock level: 220
Average weekly profit: 3124.205
Average unsold hoodies: 28.351
Average unfulfilled demand: 7.206
Percent of weeks with a stockout: 27.4%.

For stock level: 260
Average weekly profit: 2401.495
Average unsold hoodies: 62.189
Average unfulfilled demand: 1.044
Percent of weeks with a stockout: 6.1%.



## Part C: Comparing Inventory Strategies (10 Points)

### Summary Table
Summarize the findings in a table with the columns: Inventory, Avg. Profit (\$), Avg. Unsold, Avg. Unfulfilled, Stockout (%).

In [10]:
# C - Summary table
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False, justify='center'))

 Stock Level  Profit   Unsold Inventory  Unfulfilled Sales  Stockout Percent
    180      3229.920        8.224            27.079              69.9      
    220      3124.205       28.351             7.206              27.4      
    260      2401.495       62.189             1.044               6.1      


### C.1
Which inventory level produces the highest average profit?

**Answer:** The inventory level that produces the highest profit is at 180 units.

### C.2
Which inventory level produces the lowest stockout probability?

**Answer:** The lowest stockout probability is at 260 units.

### C.3
Which inventory level has the greatest average number of unsold hoodies?

**Answer:** The highest average of unsold hoodies is at 260 units.

### C.4
Why might the inventory level with the lowest stockout probability not be the most profitable option?

**Answer:** The inventory level with the lowest stockout probability may also carry a larger amount of inventory at the end of each wek. As unsold inventory has no salvage value in the analysis, leftover inventory represents loss of revenue per hoodie that can lead to lower profitability compared to a lower inventory level that will frequently stockout, but not have unsold inventory.

## Part D: Business Recommendation (5 Points)

Write a brief recommendation (approximately 100 words) to the bookstore manager explaining which inventory level you would choose. Support the recommendation with at least two numerical findings from the simulation, considering both profitability and inventory risk.

### Recommendation

We recommend using an inventory level 180 units to maximize profitability. At 180 unit inventory level, it is expected to average \$3229.92 per week. This is higher than the second best level of 220 units at \$3124.21 per week. At 180 units, there is also a risk of stockout 69.9% of the weeks but this is still the best selection due to the model not carrying inventory. 260 units is the lowest profitability based on the high average unsold inventory of 62.19 units per week. This is a significant loss of revenue.